# VRSBench Image Captioning — SmolVLM2-2.2B-Instruct + LoRA + Length-Penalty Loss
All installs, dataset loading, and model/processor downloads happen in Section 1 so nothing pulls
data mid-run. LoRA finetune (rank 16/alpha 16) on 3.5K images, with an explicit length-penalty loss
term that penalizes deviation from the dataset's median caption length (53 words) — separate from,
and in addition to, the truncation-based length control used in earlier runs. Final model is saved
to `/kaggle/working/` (the Kaggle output directory). Eval on 50 official val images. No time budget
constraint this run.

## 1. Setup — installs, imports, data + model download (everything up front)

In [1]:
!pip install -q transformers accelerate peft bitsandbytes trl num2words
!pip install -q sentence-transformers nltk rouge-score pycocoevalcap


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 17.2 MB/s eta 0:00:00


In [2]:
import os
import json
import random
import time

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

RUN_START = time.time()
def elapsed_min():
    return (time.time() - RUN_START) / 60


Device: cuda


In [3]:
# --- Dataset paths (mounted Kaggle input, no download needed) ---
DATA_DIR = "/kaggle/input/datasets/ayaanmustafa/vrs-bench/vrsbench_data"
ANNOT_FILE = os.path.join(DATA_DIR, "VRSBench_train_caption.json")
IMG_DIR = os.path.join(DATA_DIR, "Images_train", "Images_train")
VAL_IMG_DIR = os.path.join(DATA_DIR, "Images_val", "Images_val")
EVAL_CAP_JSON = os.path.join(DATA_DIR, "VRSBench_EVAL_Cap.json")

# --- Output directory (Kaggle persists /kaggle/working/ as notebook output) ---
OUTPUT_DIR = "/kaggle/working/smolvlm_vrsbench_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [4]:
# --- Model + processor download (pulled once here, used by every later cell) ---
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_NAME, do_image_splitting=False)
# do_image_splitting=False: SmolVLM's default tiling expands each image into multiple sub-tile crops,
# each with its own token block — with 3 images (2 few-shot + query) that's what blew past 4096 tokens.
# Disabling tiling uses one token block per image instead, cutting per-sample length drastically.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Model + processor downloaded. Elapsed: {elapsed_min():.1f} min")


processor_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Model + processor downloaded. Elapsed: 1.7 min


## 2. Build DataFrame from train captions

In [5]:
DATA_DIR = "/kaggle/input/datasets/ayaanmustafa/vrs-bench/vrsbench_data"
ANNOT_FILE = os.path.join(DATA_DIR, "VRSBench_train.json")  # update filename if different
IMG_DIR = os.path.join(DATA_DIR, "Images_train", "Images_train")

with open(ANNOT_FILE, "r") as f:
    raw = json.load(f)

caption_rows = []

for item in raw:
    conversations = item["conversations"]

    if "[caption]" in conversations[0]["value"].lower():
        caption_rows.append({
            "id": item["id"],
            "image": item["image"],
            "caption": conversations[1]["value"]
        })

df = pd.DataFrame(caption_rows)
# Normalize columns: expect a filename column and a caption column
# (adjust the renames below to match the printed column names above)
rename_map = {}
for c in df.columns:
    lc = c.lower()
    if "image" in lc and "path" not in lc:
        rename_map[c] = "filename"
    elif lc in ("image_path", "path"):
        rename_map[c] = "filename"
    elif "caption" in lc:
        rename_map[c] = "caption"

df = df.rename(columns=rename_map)
df["image_path"] = df["filename"].apply(lambda x: os.path.join(IMG_DIR, os.path.basename(str(x))))

df = df[["image_path", "caption"]].dropna().reset_index(drop=True)
print(df.shape)
df.head()


(20264, 2)


,image_path,caption
0,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a r..."
1,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,This aerial image from Google Earth shows a de...
2,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."
3,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a f..."
4,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."


## 3. Train subset — 3.5K images

In [6]:
from sklearn.model_selection import train_test_split

train_df, _ = train_test_split(df, test_size=0.1, random_state=SEED)

TRAIN_SUBSET = 3500   # can bump higher — training now stops on wall-clock time, not epoch count, so a bigger
                       # pool just means each epoch takes longer / fewer full passes finish in the window
train_df = train_df.sample(n=min(TRAIN_SUBSET, len(train_df)), random_state=SEED).reset_index(drop=True)

FEWSHOT_IDX = [0, 1]
fewshot_examples = train_df.iloc[FEWSHOT_IDX].reset_index(drop=True)
train_df = train_df.drop(index=FEWSHOT_IDX).reset_index(drop=True)

print("Train:", len(train_df), "| Few-shot examples:", len(fewshot_examples))


Train: 3498 | Few-shot examples: 2


## 4. LoRA setup
Rank 16 / alpha 16 on the language-model attention/MLP projections; vision tower frozen.

In [7]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 21,073,920 || all params: 2,267,858,800 || trainable%: 0.9292


## 5. Metric-aware prompt with image few-shot
Median caption length in VRSBench is 53 words — stated explicitly, anchored by 2 real image
examples at approximately that length.

In [8]:
MEDIAN_WORDS = 53

SYSTEM_PROMPT = (
    "You are a remote-sensing image captioning assistant. "
    f"Describe the satellite/aerial image in ONE concise paragraph of about {MEDIAN_WORDS} words. "
    "Mention key objects, their approximate spatial layout (e.g. top-left, bottom-right, center), "
    "and scene context. Do not hedge, apologize, or comment on image quality. Do not repeat phrases."
)

def build_conversation(image, caption=None, use_fewshot=True):
    content = [{"type": "text", "text": SYSTEM_PROMPT + "\n\nExamples of the target style and length:"}]
    if use_fewshot:
        for _, row in fewshot_examples.iterrows():
            ex_image = Image.open(row["image_path"]).convert("RGB")
            content.append({"type": "image", "image": ex_image})
            content.append({"type": "text", "text": f"Caption: {row['caption']}"})
    content.append({"type": "text", "text": "\nNow caption this image:"})
    content.append({"type": "image", "image": image})

    messages = [{"role": "user", "content": content}]
    if caption is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": caption}]})
    return messages


## 6. PyTorch Dataset
Targets are NOT hard-truncated to a single fixed length here — the point of this run is to let the
length-penalty loss (Section 8) do that work via gradient signal instead of blunt truncation, so
the model learns to converge toward the median rather than just being clipped at a wall. A loose
safety cap (90 words) still applies so no single outlier caption dominates a batch.

In [9]:
from torch.utils.data import Dataset

class VRSBenchVLMDataset(Dataset):
    def __init__(self, df, use_fewshot=True):
        self.df = df
        self.use_fewshot = use_fewshot

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        caption = row["caption"]
        words = caption.split()
        if len(words) > 90:   # loose safety cap only, not the primary length-control mechanism
            caption = " ".join(words[:90])
        messages = build_conversation(image, caption, use_fewshot=self.use_fewshot)
        return {"messages": messages}


train_dataset = VRSBenchVLMDataset(train_df, use_fewshot=True)
print(len(train_dataset))


3498


## 7. Collator

In [10]:
def collate_fn(examples):
    texts = [processor.apply_chat_template(ex["messages"], tokenize=False) for ex in examples]
    images = [[c["image"] for c in msg["content"] if c["type"] == "image"]
              for ex in examples for msg in ex["messages"] if msg["role"] == "user"]

    # No truncation at all: with do_image_splitting=False, sequences should now comfortably fit in
    # memory, and truncation is what caused the image-token mismatch twice already — safer to let
    # padding alone handle batching and never risk slicing through an image's token block.
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    return batch


## 8. Length-penalty loss — penalizes deviation from the median (53 words)
Standard cross-entropy loss is teacher-forced against the (mostly unmodified) reference caption, so
it doesn't by itself push the model toward a specific target length. This adds an explicit auxiliary
term: for each sample, count the non-masked label tokens (the response length actually being
trained on) and penalize its squared deviation from a target token count corresponding to
`MEDIAN_WORDS`. Unlike a hard truncation cap, this is a smooth gradient signal that pulls both
long AND short captions toward the median, not just clipping long ones.

In [11]:
from transformers import Trainer

# Rough token-per-word ratio for this tokenizer, used to convert the 53-word target into a token-count target
SAMPLE_TEXT = " ".join(["word"] * 53)
TARGET_TOKEN_COUNT = len(processor.tokenizer(SAMPLE_TEXT)["input_ids"])
print("Target token count for median length:", TARGET_TOKEN_COUNT)

LENGTH_PENALTY_LAMBDA = 0.02   # weight of the length term relative to CE loss — small, a nudge not a takeover

class LengthPenalizedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        ce_loss = outputs.loss

        labels = inputs["labels"]
        response_lengths = (labels != -100).sum(dim=-1).float()  # tokens actually trained on, per sample

        deviation = (response_lengths - TARGET_TOKEN_COUNT) ** 2
        length_loss = deviation.mean() / (TARGET_TOKEN_COUNT ** 2)   # normalized so it stays a small, stable term

        loss = ce_loss + LENGTH_PENALTY_LAMBDA * length_loss

        if self.state.global_step % 20 == 0:
            print(f"  [step {self.state.global_step}] ce={ce_loss.item():.4f} "
                  f"len_penalty={length_loss.item():.4f} avg_response_tokens={response_lengths.mean().item():.1f}")

        return (loss, outputs) if return_outputs else loss


Target token count for median length: 53


## 9. Training — wall-clock capped to a 4-5hr session
Per-step throughput on this hardware isn't known ahead of time, so instead of guessing an epoch
count that might finish in 40 minutes or blow past 8 hours, `num_train_epochs` is set high as an
upper bound and a callback stops training gracefully once the target wall-clock time is reached —
whichever comes first. This reliably fills the available session instead of under- or over-shooting.

In [12]:
from transformers import TrainingArguments, TrainerCallback

TARGET_TRAIN_MINUTES = 270   # ~4.5hr — middle of the 4-5hr window, leaves buffer for save/eval after

class TimeLimitCallback(TrainerCallback):
    def __init__(self, target_minutes):
        self.target_seconds = target_minutes * 60
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time
        if elapsed >= self.target_seconds:
            print(f"\n[TimeLimitCallback] Reached {elapsed/60:.1f} min — stopping training gracefully "
                  f"(completed {state.global_step} steps, epoch {state.epoch:.2f}).")
            control.should_training_stop = True
        return control

use_bf16 = torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir="/kaggle/working/smolvlm_vrsbench_checkpoints",
    per_device_train_batch_size=1,   # lowered — 3 images (2 few-shot + query) per sample at up to 4096 tokens
                                      # is much heavier than a single-image batch; safer for T4-class memory
    gradient_accumulation_steps=16,  # effective batch still 16
    num_train_epochs=50,             # high upper bound — the time callback will stop training first
    learning_rate=2e-4,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=20,
    save_strategy="steps",
    save_steps=100,                  # periodic saves in case the session gets interrupted before the time cap
    save_total_limit=3,
    bf16=use_bf16,
    fp16=not use_bf16,
    remove_unused_columns=False,
    report_to="none",
    gradient_checkpointing=True,
)

trainer = LengthPenalizedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    callbacks=[TimeLimitCallback(TARGET_TRAIN_MINUTES)],
)

train_result = trainer.train()
print(train_result)
print(f"Training took {elapsed_min():.1f} min total (incl. setup)")


  [step 0] ce=11.3653 len_penalty=79.6472 avg_response_tokens=526.0


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at /pytorch/torch/csrc/autograd/input_buffer.cpp:240.)
  return Variable._execution_en

  [step 0] ce=11.5855 len_penalty=77.6394 avg_response_tokens=520.0
  [step 0] ce=12.5195 len_penalty=62.2015 avg_response_tokens=471.0
  [step 0] ce=12.2137 len_penalty=67.3638 avg_response_tokens=488.0
  [step 0] ce=11.7819 len_penalty=72.7319 avg_response_tokens=505.0
  [step 0] ce=12.2378 len_penalty=64.9089 avg_response_tokens=480.0
  [step 0] ce=11.5955 len_penalty=76.9758 avg_response_tokens=518.0
  [step 0] ce=11.7340 len_penalty=73.7006 avg_response_tokens=508.0
  [step 0] ce=11.6022 len_penalty=74.3499 avg_response_tokens=510.0
  [step 0] ce=11.2365 len_penalty=82.0221 avg_response_tokens=533.0
  [step 0] ce=11.5346 len_penalty=77.9722 avg_response_tokens=521.0
  [step 0] ce=11.1717 len_penalty=82.0221 avg_response_tokens=533.0
  [step 0] ce=11.1454 len_penalty=84.7789 avg_response_tokens=541.0
  [step 0] ce=11.5049 len_penalty=78.3058 avg_response_tokens=522.0
  [step 0] ce=11.6354 len_penalty=74.0249 avg_response_tokens=509.0
  [step 0] ce=10.8219 len_penalty=89.3560 avg_re

Step,Training Loss
20,203.879370
40,126.205530
60,37.438177
80,28.700687
100,28.275015
120,27.934320


  [step 20] ce=10.0503 len_penalty=76.3151 avg_response_tokens=516.0
  [step 20] ce=10.0624 len_penalty=74.3499 avg_response_tokens=510.0
  [step 20] ce=10.2260 len_penalty=72.7319 avg_response_tokens=505.0
  [step 20] ce=10.1956 len_penalty=72.4105 avg_response_tokens=504.0
  [step 20] ce=10.3130 len_penalty=71.4503 avg_response_tokens=501.0
  [step 20] ce=9.8568 len_penalty=77.9722 avg_response_tokens=521.0
  [step 20] ce=10.4655 len_penalty=67.0545 avg_response_tokens=487.0
  [step 20] ce=9.7586 len_penalty=82.0221 avg_response_tokens=533.0
  [step 20] ce=10.1668 len_penalty=72.0897 avg_response_tokens=503.0
  [step 20] ce=10.1608 len_penalty=72.7319 avg_response_tokens=505.0
  [step 20] ce=10.3294 len_penalty=71.1317 avg_response_tokens=500.0
  [step 20] ce=9.8062 len_penalty=78.3058 avg_response_tokens=522.0
  [step 20] ce=9.4920 len_penalty=90.4293 avg_response_tokens=557.0
  [step 20] ce=10.2289 len_penalty=71.7697 avg_response_tokens=502.0
  [step 20] ce=10.3279 len_penalty=68.

## 10. Save final model to the output directory

In [13]:
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Saved final model + processor to:", OUTPUT_DIR)
!ls {OUTPUT_DIR}


Saved final model + processor to: /kaggle/working/smolvlm_vrsbench_final
adapter_config.json	   processor_config.json  tokenizer.json
adapter_model.safetensors  README.md
chat_template.jinja	   tokenizer_config.json


## 11. Metric suite

In [14]:
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

bert_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

def get_ngrams(tokens, n):
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def bert_bleu4(candidate, reference, alpha=0.5):
    c_tokens = candidate.lower().split()
    r_tokens = reference.lower().split()
    c_ngrams_by_n = [get_ngrams(c_tokens, n) for n in range(1, 5)]
    r_ngrams_by_n = [get_ngrams(r_tokens, n) for n in range(1, 5)]
    all_texts = [g for grams in c_ngrams_by_n + r_ngrams_by_n for g in grams]
    if not all_texts:
        return 0.0
    embs = bert_model.encode(all_texts, convert_to_tensor=True, show_progress_bar=False, batch_size=256)
    embs = F.normalize(embs, dim=-1)
    idx = 0
    c_embs_by_n, r_embs_by_n = [], []
    for grams in c_ngrams_by_n:
        c_embs_by_n.append(embs[idx: idx + len(grams)]); idx += len(grams)
    for grams in r_ngrams_by_n:
        r_embs_by_n.append(embs[idx: idx + len(grams)]); idx += len(grams)
    log_p_sum = 0.0
    for n in range(4):
        C_emb, R_emb = c_embs_by_n[n], r_embs_by_n[n]
        if len(C_emb) == 0 or len(R_emb) == 0:
            p_n = 1e-9
        else:
            sims = torch.mm(R_emb, C_emb.T)
            p_n = max(sims.max(dim=1).values.mean().item(), 1e-9)
        log_p_sum += np.log(p_n)
    L_c, L_r = len(c_tokens), max(len(r_tokens), 1)
    lp = np.exp(-alpha * abs(L_c - L_r) / L_r)
    return lp * np.exp(log_p_sum / 4)

def corpus_bert_bleu4(candidates, references):
    return float(np.mean([bert_bleu4(c, r) for c, r in zip(candidates, references)]))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider

smoothie = SmoothingFunction().method4
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def compute_bleu_n(candidates, references, n):
    weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
    scores = []
    for c, r in zip(candidates, references):
        c_tok, r_tok = c.lower().split(), [r.lower().split()]
        scores.append(sentence_bleu(r_tok, c_tok, weights=weights, smoothing_function=smoothie))
    return float(np.mean(scores))

def compute_meteor(candidates, references):
    return float(np.mean([meteor_score([r.lower().split()], c.lower().split()) for c, r in zip(candidates, references)]))

def compute_rouge_l(candidates, references):
    return float(np.mean([rouge.score(r, c)["rougeL"].fmeasure for c, r in zip(candidates, references)]))

def compute_cider(candidates, references):
    gts = {i: [r] for i, r in enumerate(references)}
    res = {i: [c] for i, c in enumerate(candidates)}
    score, _ = Cider().compute_score(gts, res)
    return float(score)

def compute_avg_len(candidates):
    return float(np.mean([len(c.split()) for c in candidates]))

def compute_all_metrics(candidates, references):
    return {
        "BLEU-1": compute_bleu_n(candidates, references, 1),
        "BLEU-2": compute_bleu_n(candidates, references, 2),
        "BLEU-3": compute_bleu_n(candidates, references, 3),
        "BLEU-4": compute_bleu_n(candidates, references, 4),
        "METEOR": compute_meteor(candidates, references),
        "ROUGE_L": compute_rouge_l(candidates, references),
        "CIDEr": compute_cider(candidates, references),
        "Avg_L": compute_avg_len(candidates),
        "BERT-BLEU4": corpus_bert_bleu4(candidates, references),
    }


## 12. Load official val split, 50-sample eval

In [16]:
with open(EVAL_CAP_JSON, "r") as f:
    eval_raw = json.load(f)

eval_df = pd.DataFrame(eval_raw)
if "type" in eval_df.columns:
    eval_df = eval_df[eval_df["type"] == "caption"].reset_index(drop=True)

eval_rename = {}
for c in eval_df.columns:
    lc = c.lower()
    if "image" in lc and "path" not in lc:
        eval_rename[c] = "filename"
    elif lc in ("image_path", "path"):
        eval_rename[c] = "filename"
    elif "caption" in lc or lc in ("ground_truth", "answer"):
        eval_rename[c] = "caption"

eval_df = eval_df.rename(columns=eval_rename)
eval_df["image_path"] = eval_df["filename"].apply(lambda x: os.path.join(VAL_IMG_DIR, os.path.basename(str(x))))
eval_df = eval_df[["image_path", "caption"]].dropna().reset_index(drop=True)

EVAL_SUBSET_SIZE = 50
eval_subset_df = eval_df.sample(n=min(EVAL_SUBSET_SIZE, len(eval_df)), random_state=SEED).reset_index(drop=True)
print("Eval samples used:", len(eval_subset_df))


Eval samples used: 50


## 13. Inference
No hard word-count truncation here — the point is to see what the length-penalty loss actually
learned on its own. `max_new_tokens` is set generously above the target so the model isn't clipped
by the decoding wall; if it overshoots, that tells us the loss weight needs to go up.

In [17]:
model.eval()

@torch.no_grad()
def generate_caption(image, max_new_tokens=120):
    messages = build_conversation(image, caption=None, use_fewshot=True)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images = [c["image"] for msg in messages for c in msg["content"] if c["type"] == "image"]

    inputs = processor(text=text, images=images, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
    )
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    text_out = processor.batch_decode(generated, skip_special_tokens=True)[0]
    return text_out.strip()


def validate(eval_df):
    preds, refs = [], []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="val"):
        image = Image.open(row["image_path"]).convert("RGB")
        pred = generate_caption(image)
        preds.append(pred)
        refs.append(row["caption"])
    metrics = compute_all_metrics(preds, refs)
    return metrics, preds, refs


metrics, preds, refs = validate(eval_subset_df)
print(metrics)
print(f"Avg predicted length: {np.mean([len(p.split()) for p in preds]):.1f} words (target median: {MEDIAN_WORDS})")
print(f"Total run time: {elapsed_min():.1f} min")


val:   0%|          | 0/50 [00:00<?, ?it/s]

{'BLEU-1': 0.19667619965206815, 'BLEU-2': 0.07794685606297148, 'BLEU-3': 0.035481292004876236, 'BLEU-4': 0.019890751221824356, 'METEOR': 0.19291462557422803, 'ROUGE_L': 0.18946293507162107, 'CIDEr': 0.07215104982238123, 'Avg_L': 63.16, 'BERT-BLEU4': 0.4907769091521302}
Avg predicted length: 63.2 words (target median: 53)
Total run time: 279.5 min


## 14. Sample predictions

In [18]:
for p, r in list(zip(preds, refs))[:5]:
    print(f"PRED ({len(p.split())}w):", p)
    print(f"REF  ({len(r.split())}w):", r)
    print("-" * 60)


PRED (81w): A high-resolution image captured via Google Earth features a large ship positioned at the left side of the frame near what appears to be harbor facilities. The vessel has distinct markings that include various colors such as white, green, blue, red, yellow, orange, gray, black, brown, light purple, dark pink, and teal. There's also some text present within its structure but it isn't clear enough for precise identification. Surrounding areas show multiple harbors connected through docks extending into the water body.
REF  (24w): The image from GoogleEarth shows a docking area with a ship. The ship has a white and green color scheme and is very long.
------------------------------------------------------------
PRED (84w): This high-resolution image captured via Google Earth features two recreational facilities within what appears to be a residential area. On the left side of the frame stands a large tennis court adjacent to another building that has a green roof. Adjacent to 